In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
!unzip -q -o "/content/drive/MyDrive/PCGITA.zip" -d "/content/"


In [ ]:
import os

print(os.listdir("/content/PCGITA"))

In [ ]:
print(os.listdir("/content/PCGITA/Vowels"))

In [ ]:
!pip install -q transformers torchaudio scikit-learn matplotlib seaborn pandas umap-learn


In [ ]:

import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import Wav2Vec2Processor, Wav2Vec2Model

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA


In [ ]:
# STEP 3 — Load PC-GITA dataset

import os
import pandas as pd

DATASET_PATH = "/content/PCGITA/Vowels"

data = []

# -------------------------
# Healthy speakers
# -------------------------

control_path = os.path.join(DATASET_PATH, "Control")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(control_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Control", vowel, file),
                "label": 0
            })


# -------------------------
# Parkinson speakers
# -------------------------

pd_path = os.path.join(DATASET_PATH, "Patologicas")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(pd_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Patologicas", vowel, file),
                "label": 1
            })


labels_df = pd.DataFrame(data)

print("Total samples:", len(labels_df))
print("\nClass distribution:")
print(labels_df["label"].value_counts())

labels_df.head()

In [ ]:
# Class distribution
sns.countplot(x="label", data=labels_df)
plt.title("Class Distribution (0 = Healthy, 1 = Parkinson)")
plt.show()

# Audio duration distribution (sample)
durations = []
for file in labels_df["filename"][:50]:
    waveform, sr = torchaudio.load(os.path.join(DATASET_PATH, file))
    durations.append(waveform.shape[1] / sr)

plt.hist(durations, bins=20)
plt.title("Audio Duration Distribution (Sample)")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()


In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model_wav = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

model_wav.eval()
for param in model_wav.parameters():
    param.requires_grad = False

In [ ]:
def extract_features_seq(audio_path):
    waveform, sr = torchaudio.load(audio_path)

    #  Fix stereo → mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    #  Fix resampling
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    try:
        inputs = processor(
            waveform.squeeze().numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model_wav(inputs.input_values)

        hidden = outputs.last_hidden_state.squeeze(0)  # (T, 768)

        return hidden

    except Exception as e:
        print("Model error:", e)
        return None

In [ ]:
X_seq = []
y = []

for i, row in labels_df.iterrows():
    path = os.path.join(DATASET_PATH, row["filename"])

    try:
        feat = extract_features_seq(path)

        if feat is None:
            print(f"Skipped (None): {path}")
            continue

        feat = feat.cpu().numpy().astype(np.float16)

        X_seq.append(feat)
        y.append(row["label"])

    except Exception as e:
        print(f" Error in file: {path}")
        print(e)

In [ ]:
np.save("X_seq.npy", np.array(X_seq, dtype=object))
np.save("y.npy", y)



In [ ]:
X_seq = np.load("X_seq.npy", allow_pickle=True)
y = np.load("y.npy")

In [ ]:
X_train_seq, X_test_seq, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.2, stratify=y
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

X_train = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_train_seq],
    batch_first=True
)

X_test = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_test_seq],
    batch_first=True
)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

#  IMPORTANT (accuracy + memory balance)
MAX_LEN = 400
X_train = X_train[:, :MAX_LEN, :]
X_test  = X_test[:, :MAX_LEN, :]

#  NORMALIZATION (BIG BOOST)
X_train = (X_train - X_train.mean()) / (X_train.std() + 1e-5)
X_test  = (X_test - X_test.mean()) / (X_test.std() + 1e-5)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

In [ ]:
import torch.nn as nn

class GRUFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        self.gru = nn.GRU(
            input_size=768,
            hidden_size=256,   #  increased
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        out, _ = self.gru(x)        # (B, T, 256)

        # 🔥 IMPORTANT CHANGE (better than last timestep)
        out = torch.mean(out, dim=1)   # (B, 256)

        logits = self.fc(out)
        prob = torch.sigmoid(logits)

        return out, prob

In [ ]:
model = GRUFeatureExtractor()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch_X, batch_y in loader:

        features, outputs = model(batch_X)

        outputs = outputs.squeeze()
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")

In [ ]:
model.eval()

with torch.no_grad():
    train_features, _ = model(X_train)
    test_features, _ = model(X_test)

train_features = train_features.numpy()
test_features = test_features.numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

In [ ]:
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

In [ ]:
svm = SVC(kernel='rbf', C=10, gamma='scale')

svm.fit(train_features, y_train_np)

y_pred = svm.predict(test_features)

print("Final Accuracy (GRU + SVM):", accuracy_score(y_test_np, y_pred))